In [13]:
import os
import time
import requests
import pandas as pd
import urllib3

# Silencia os avisos de SSL no terminal/notebook
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

output_folder = "gutenberg_harvest"
os.makedirs(output_folder, exist_ok=True)

# 1. Carrega os IDs do CSV para saber EXATAMENTE o que baixar
df_meta_source = pd.read_csv("gutenberg_books.csv", dtype={"book_id": str})
target_ids = df_meta_source["book_id"].dropna().str.strip().tolist()

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

MAX_DOWNLOADS = 20  # Quantidade de livros para baixar do CSV
downloaded = 0

print(f"Iniciando download baseado nos IDs do CSV (Meta: {MAX_DOWNLOADS} livros)...\n")

for book_id in target_ids:
    if downloaded >= MAX_DOWNLOADS:
        break
        
    # Verifica se o arquivo HTML ou ZIP já existe localmente
    already_downloaded = any(
        f.startswith(f"{book_id}-h") or f.startswith(f"pg{book_id}") 
        for f in os.listdir(output_folder)
    )
    if already_downloaded:
        downloaded += 1
        print(f"[{downloaded}/{MAX_DOWNLOADS}] Livro ID {book_id} já existe na pasta. Pulando...")
        continue

    # Tenta as URLs padrões do Project Gutenberg para o ID
    urls_to_try = [
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-h.zip",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-h.htm",
        f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}-images.html"
    ]
    
    success = False
    for url in urls_to_try:
        try:
            r = requests.get(url, headers=headers, timeout=12, verify=False)
            if r.status_code == 200 and len(r.content) > 3000:
                ext = ".zip" if url.endswith(".zip") else ".htm"
                save_path = os.path.join(output_folder, f"{book_id}-h{ext}")
                with open(save_path, 'wb') as f:
                    f.write(r.content)
                
                downloaded += 1
                print(f"[{downloaded}/{MAX_DOWNLOADS}] Baixado ID {book_id} com sucesso!")
                success = True
                time.sleep(1)
                break
        except Exception:
            continue
            
    if not success:
        print(f"Não foi possível baixar o ID {book_id} pelas URLs padrão.")

print(f"\nDownload concluído! {downloaded} livros sincronizados com o CSV.")

Iniciando download baseado nos IDs do CSV (Meta: 20 livros)...

[1/20] Baixado ID 1342 com sucesso!
[2/20] Baixado ID 2701 com sucesso!
[3/20] Baixado ID 2554 com sucesso!
[4/20] Baixado ID 84 com sucesso!
[5/20] Baixado ID 11 com sucesso!
[6/20] Baixado ID 345 com sucesso!
[7/20] Baixado ID 43 com sucesso!
[8/20] Baixado ID 2641 com sucesso!
[9/20] Baixado ID 145 com sucesso!
[10/20] Baixado ID 65238 com sucesso!
[11/20] Baixado ID 3268 com sucesso!
[12/20] Baixado ID 67979 com sucesso!
[13/20] Baixado ID 37106 com sucesso!
[14/20] Baixado ID 1260 com sucesso!
[15/20] Baixado ID 2465 com sucesso!
[16/20] Baixado ID 59828 com sucesso!
[17/20] Baixado ID 2868 com sucesso!
[18/20] Baixado ID 244 com sucesso!
[19/20] Baixado ID 45839 com sucesso!
[20/20] Baixado ID 62215 com sucesso!

Download concluído! 20 livros sincronizados com o CSV.


In [14]:
import os
import re
import glob
from bs4 import BeautifulSoup
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

nltk.download('stopwords')
nltk.download('punkt')

def extract_text_from_html(file_path):
    """
    Realiza o scraping do arquivo HTML baixado do livro, 
    removendo tags HTML, cabeçalhos/rodapés e extraindo o texto limpo do conteúdo.
    """
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    # Remove scripts, estilos e cabeçalhos
    for script in soup(["script", "style", "header", "footer"]):
        script.extract()

    # Extrai texto do corpo da página
    text = soup.get_text(separator=" ")

    # Limpeza de marcas padrão do Project Gutenberg (Header e Footer da licença)
    start_match = re.search(r"\*\*\*\s*START OF TH(IS|E) PROJECT GUTENBERG EBOOK.*?\*\*\*", text, re.IGNORECASE)
    end_match = re.search(r"\*\*\*\s*END OF TH(IS|E) PROJECT GUTENBERG EBOOK.*?\*\*\*", text, re.IGNORECASE)

    if start_match and end_match:
        text = text[start_match.end():end_match.start()]
    elif start_match:
        text = text[start_match.end():]

    # Normalização básica do texto
    text = re.sub(r"\s+", " ", text).strip()
    return text

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\joaoe\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\joaoe\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [15]:
import zipfile

def build_dataset_from_harvest(base_folder):
    """
    Descompacta arquivos .zip da pasta harvest e varre os arquivos HTML 
    para construir o DataFrame de livros.
    """
    # 1. Extrai todos os arquivos .zip baixados para a pasta local
    zip_files = glob.glob(os.path.join(base_folder, "*.zip"))
    print(f"Descompactando {len(zip_files)} arquivos .zip...")
    for zip_path in zip_files:
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(base_folder)
        except Exception as e:
            print(f"Erro ao descompactar {zip_path}: {e}")

    # 2. Varre todos os HTMLs gerados na descompactação
    html_files = glob.glob(os.path.join(base_folder, "**/*.htm*"), recursive=True)
    
    data = []
    print(f"Total de arquivos HTML encontrados: {len(html_files)}")

    for file_path in html_files:
        filename = os.path.basename(file_path)
        # Extrai ID do livro a partir do nome do arquivo (ex: 1342-h.htm -> 1342)
        match = re.search(r"(\d+)", filename)
        if not match:
            continue
            
        book_id = match.group(1)
        
        # Faz a raspagem do conteúdo textual
        cleaned_text = extract_text_from_html(file_path)
        
        # Filtra textos muito curtos ou vazios
        if len(cleaned_text) < 1000:
            continue
            
        data.append({
            "book_id": book_id,
            "file_path": file_path,
            "text": cleaned_text
        })

    df = pd.DataFrame(data)
    return df

df_books = build_dataset_from_harvest("gutenberg_harvest")
print(f"Livros válidos processados: {len(df_books)}")
df_books.head()

Descompactando 21 arquivos .zip...
Total de arquivos HTML encontrados: 39
Livros válidos processados: 39


,book_id,file_path,text
0,11,gutenberg_harvest\11-h.htm,Alice’s Adventures in Wonderland | Project Gut...
1,1260,gutenberg_harvest\1260-h.htm,Jane Eyre | Project Gutenberg JANE EYRE AN AUT...
2,1342,gutenberg_harvest\1342-h.htm,Pride and prejudice | Project Gutenberg PREFAC...
3,145,gutenberg_harvest\145-h.htm,Middlemarch | Project Gutenberg Middlemarch Ge...
4,244,gutenberg_harvest\244-h.htm,A Study in Scarlet | Project Gutenberg A STUDY...


In [16]:
# Mapeamento simples de palavras-chave dos Subjects para gêneros simplificados
def map_subject_to_genre(subject):
    if not isinstance(subject, str):
        return "General Fiction"
    
    subject_lower = subject.lower()
    if "science fiction" in subject_lower:
        return "Science Fiction"
    elif "detective" in subject_lower or "mystery" in subject_lower:
        return "Detective & Mystery"
    elif "horror" in subject_lower or "gothic" in subject_lower:
        return "Horror"
    elif "love" in subject_lower or "romance" in subject_lower:
        return "Romance"
    elif "adventure" in subject_lower or "sea stories" in subject_lower:
        return "Adventure"
    else:
        return "Fiction"

# Carrega metadados salvos previamente para obter os rótulos de treino
# Garante que ambos estejam como string e sem espaços nas pontas
df_books["book_id"] = df_books["book_id"].astype(str).str.strip()
df_meta["book_id"] = df_meta["book_id"].astype(str).str.strip()

# Merge
df_dataset = pd.merge(df_books, df_meta[["book_id", "title", "genre"]], on="book_id", how="inner")

print(f"Total de registros casados: {len(df_dataset)}")
print("\nDistribuição por Gênero Literário:")
print(df_dataset["genre"].value_counts())

# 1. IDs extraídos dos arquivos HTML baixados
ids_baixados = set(df_books["book_id"])
print(f"IDs nos arquivos HTML ({len(ids_baixados)}):", list(ids_baixados)[:10])

# 2. IDs presentes no arquivo CSV de metadados
ids_metadata = set(df_meta["book_id"])
print(f"IDs no CSV de metadados ({len(ids_metadata)}):", list(ids_metadata)[:10])

# 3. Quantos realmente coincidem
coincidentes = ids_baixados.intersection(ids_metadata)
print(f"\nIDs coincidentes entre os dois ({len(coincidentes)}):", coincidentes)

Total de registros casados: 21

Distribuição por Gênero Literário:
genre
Fiction                5
Detective & Mystery    5
Romance                4
Horror                 4
Science Fiction        2
Adventure              1
Name: count, dtype: int64
IDs nos arquivos HTML (39): ['1940', '1729', '10084', '67979', '2303', '1715', '1837', '2304', '1260', '2701']
IDs no CSV de metadados (281): ['768', '27475', '1212', '36725', '48893', '17157', '2527', '53416', '38250', '22342']

IDs coincidentes entre os dois (21): {'67979', '2701', '1260', '3268', '59828', '1680', '65238', '345', '1342', '244', '84', '43', '2554', '2868', '145', '62215', '2641', '2465', '45839', '37106', '11'}


In [17]:
# 1. Pré-processamento e Vetorização TF-IDF
stop_words_en = stopwords.words('english')

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words=stop_words_en,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df_dataset["text"])
y = df_dataset["genre"]

# Verifica se há amostras suficientes antes de dividir
if len(df_dataset) < 4:
    print(f"Atenção: O dataset possui apenas {len(df_dataset)} exemplo(s). Baixe mais livros para treinar o modelo.")
else:
    # 2. Divisão Treino / Teste (Corrigido: train_test_split)
    # stratify só é usado se cada classe tiver pelo menos 2 exemplos
    use_stratify = y.value_counts().min() >= 2
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=(y if use_stratify else None)
    )

    # 3. Treinamento do Classificador Naive Bayes
    model = MultinomialNB()
    model.fit(X_train, y_train)

    # 4. Avaliação do Modelo
    y_pred = model.predict(X_test)

    print("Acurácia na detecção de gênero apenas pelo conteúdo HTML:")
    print(f"{accuracy_score(y_test, y_pred):.2%}\n")
    print("Relatório de Classificação de PLN:")
    print(classification_report(y_test, y_pred))

Acurácia na detecção de gênero apenas pelo conteúdo HTML:
14.29%

Relatório de Classificação de PLN:
                     precision    recall  f1-score   support

          Adventure       0.00      0.00      0.00         1
Detective & Mystery       0.14      1.00      0.25         1
            Fiction       0.00      0.00      0.00         3
            Romance       0.00      0.00      0.00         1
    Science Fiction       0.00      0.00      0.00         1

           accuracy                           0.14         7
          macro avg       0.03      0.20      0.05         7
       weighted avg       0.02      0.14      0.04         7



c:\Users\joaoe\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\joaoe\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\joaoe\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [18]:
def predict_book_genre(html_file_path):
    """
    Recebe um caminho de arquivo HTML de um livro qualquer,
    raspa o conteúdo, aplica o vetorizador e determina o gênero literário.
    """
    raw_text = extract_text_from_html(html_file_path)
    features = vectorizer.transform([raw_text])
    predicted_genre = model.predict(features)[0]
    probabilities = model.predict_proba(features)
    
    print(f"Arquivo: {os.path.basename(html_file_path)}")
    print(f"Gênero Predito: {predicted_genre}")
    return predicted_genre

# Exemplo de teste em um arquivo do dataset:
if not df_dataset.empty:
    sample_file = df_dataset.iloc[0]["file_path"]
    predict_book_genre(sample_file)

Arquivo: 11-h.htm
Gênero Predito: Detective & Mystery
